# Training a convnet on the CIFAR-10 dataset with MindSpore

Last time we trained a simple linear neural network on the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset. The accuracy was only about $25\mathrm{-}40\%$ since the simple linear model was unable to effectively learn the high-dimensional data, a textbook example of underfitting.

This time, we'll adapt [LeNet](https://en.wikipedia.org/wiki/LeNet) with modern building blocks and train our adapted network on the same dataset. LeNet is the first example of a convolutional neural network \(CNN\) to be successfully trained on the [MNIST handwritten digits dataset](https://en.wikipedia.org/wiki/MNIST_database).

You are advised to go through the first 7 chapters of the [D2L](https://d2l.ai/chapter_convolutional-neural-networks/index.html) textbook or have equivalent experience in machine learning to follow through this notebook experiment effectively.

Here's how we will pre-process the input images for training:

1. Resize all images to $32 \times 32$ pixels - should already be the case but just to be on the safe side
1. Rescale all images by a factor of $\frac{1}{255}$ so pixel values stay within the range $[0, 1]$
1. Reorder the dimensions from NHWC to NCHW

We will also apply one-hot encoding to the class labels.

Our neural network architecture based on LeNet below.

1. 1st convolution layer:
    1. 3 input channels
    1. 32 output channels
    1. $3 \times 3$ kernel with unit stride
    1. Padding of `1px` in all directions
    1. ReLU activation
1. 1st batch normalization layer with 32 channels, epsilon = `1e-5`, momentum = 0.9
1. 1st max pooling layer with $2 \times 2$ window and a stride of 2
1. 2nd convolution layer:
    1. 32 input channels
    1. 64 output channels
    1. $3 \times 3$ kernel with unit stride
    1. Padding of `1px` in all directions
    1. ReLU activation
1. 2nd batch normalization layer with 64 channels, epsilon = `1e-5`, momentum = 0.9
1. 2nd max pooling layer with $2 \times 2$ window and a stride of 2
1. Flattening layer to convert the $64 \times 8 \times 8$ feature maps to 4096 output channels
1. 1st hidden fully connected layer: 4096 input channels, 2048 output channels, ReLU activation
1. 1st dropout layer with `p=0.5`
1. 2nd hidden fully connected layer: 2048 input channels, 1024 output channels, ReLU activation
1. 2nd dropout layer with `p=0.5`
1. Final fully connected layer: 1024 input channels, 10 output channels

Our choice of hyperparameters below.

1. Combined activation and loss function: softmax cross entropy with logits using the log-sum-exp trick
1. Optimization algorithm: minibatch SGD with batch size of 128
1. Learning rate: `0.01`
1. Weight decay: `1e-4`
1. Momentum: `0.9`
1. Epochs: max. 100 with early stopping, wait at most 5 epochs and restore model weights from the best epoch

The software versions used as below.

1. Python 3.12
1. MindSpore 2.8.0
1. CANN 8.5.0

In [1]:
!cat requirements.txt

absl-py==2.4.0
attrs==26.1.0
cloudpickle==3.1.2
decorator==5.2.1
jupyterlab==4.5.7
jupyterlab-git==0.53.0
jupyter-resource-usage==1.2.1
mindspore==2.8.0
ml-dtypes==0.5.4
sympy==1.14.0
tornado==6.5.5


In [2]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## Data loading and preprocessing

In [4]:
import os

dataset_dir = 'data/'
os.makedirs(dataset_dir, exist_ok=True)

In [5]:
import tarfile
import urllib.request

cifar10_url = 'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz'

with urllib.request.urlopen(cifar10_url) as response:
    with tarfile.open(fileobj=response, mode='r|gz') as tar:
        tar.extractall(path=dataset_dir, filter='data')

dataset_dir = os.path.join(dataset_dir, 'cifar-10-batches-bin/')
dataset_dir

'data/cifar-10-batches-bin/'

In [6]:
import mindspore.dataset as ds

train_ds = ds.Cifar10Dataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.Cifar10Dataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [7]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(32, 32)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128)
    return dataset

train_ds, test_ds = transform_ds(dataset=train_ds), transform_ds(dataset=test_ds)

## Model training and evaluation

TODO